# Phase 31 Step 311: Automated Entity Disambiguation

**Purpose**: Deterministic alignment bootstrap across all sources for 7 core classes.

**Input**: Broadcasting programs, mention detection CSVs, Wikidata/Fernsehserien projections  
**Output**: `aligned_*.csv` files per core class in `data/31_entity_disambiguation/`

**Key Principles**:
- Layer 1-4 matching model (broadcasts → episodes → persons → roles/organizations)
- Precision-first: never force match below confidence threshold
- Event-sourced: all decisions logged for reproducibility and recovery
- Deterministic: same input always produces identical output
- One person mention in one episode = canonical matching unit

**Runtime**: Single `Run All` deterministic execution. No manual edits. Replayable from checkpoints.

In [1]:
"""
Bootstrap: Discover repository root and set up import paths.
This cell runs first and must not depend on project modules.
"""
import sys
from pathlib import Path

# Find repo root by walking up from notebook location
notebook_dir = Path.cwd()
repo_root = None

for parent in [notebook_dir] + list(notebook_dir.parents):
    if (parent / "speakermining" / "src" / "process").exists():
        repo_root = parent
        break

if not repo_root:
    raise RuntimeError(
        f"Could not find repository root from {notebook_dir}. "
        "Expected 'speakermining/src/process' folder structure."
    )

# Add src to path for process module imports
src_path = repo_root / "speakermining" / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Repository root: {repo_root}")
print(f"Import path added: {src_path}")
print(f"Working directory: {Path.cwd()}")

Repository root: c:\workspace\git\borgnetzwerk\speaker-mining
Import path added: c:\workspace\git\borgnetzwerk\speaker-mining\speakermining\src
Working directory: c:\workspace\git\borgnetzwerk\speaker-mining\speakermining\src\process\notebooks


In [2]:
# Import project modules
import pandas as pd
from datetime import datetime, timezone
import shutil

from process.entity_disambiguation import (
    Step311Orchestrator,
    RecoveryOrchestrator,
)
from process.entity_disambiguation.config import PHASE_DIR, get_aligned_csv_path, CORE_CLASSES

PROJECTION_ROOT = PHASE_DIR / "projections"

print("✓ All imports successful")

✓ All imports successful


## Step 1: Check for Recovery Checkpoint

If this notebook was interrupted, detect and restore from the most recent checkpoint.

In [3]:
# Check for recovery checkpoint
print("Checking for recovery checkpoint...")

# Set to True to wipe local Step 311 state before running.
# This removes event-store chunks, checkpoints, handler progress DB,
# and the derived projections folder under data/31_entity_disambiguation.
RESET_STATE = False

# Set to True only when you want to bypass recovery loading entirely.
FORCE_FRESH_RUN = False

if RESET_STATE:
    print("✓ Clean-slate reset enabled; wiping Step 311 local state")

    events_dir = PHASE_DIR / "events"
    checkpoints_dir = PHASE_DIR / "checkpoints"
    handler_progress_db = PHASE_DIR / "handler_progress.db"
    projection_root = PHASE_DIR / "projections"

    removed = []

    if events_dir.exists():
        shutil.rmtree(events_dir)
        removed.append(str(events_dir))

    if checkpoints_dir.exists():
        shutil.rmtree(checkpoints_dir)
        removed.append(str(checkpoints_dir))

    if handler_progress_db.exists():
        handler_progress_db.unlink()
        removed.append(str(handler_progress_db))

    if projection_root.exists():
        shutil.rmtree(projection_root)
        removed.append(str(projection_root))

    if removed:
        print("  Removed artifacts:")
        for p in removed:
            print(f"  - {p}")
    else:
        print("  Nothing to remove (state already clean)")

if FORCE_FRESH_RUN or RESET_STATE:
    print("✓ Recovery restore disabled for this run")
    recovered_projections = None
else:
    recovery_orchestrator = RecoveryOrchestrator()
    recovered_projections = recovery_orchestrator.recover_and_resume()

if recovered_projections:
    print("✓ Recovery checkpoint found and validated")
    print(f"  Recovered projections available: {list(recovered_projections.keys())}")
    print("✓ Proceeding with a full fresh import pass to detect new CSV content")
else:
    print("✓ No recovery checkpoint used; starting fresh run")

Checking for recovery checkpoint...
✓ Recovery checkpoint found and validated
  Recovered projections available: ['broadcasting_programs', 'series', 'episodes', 'persons', 'topics', 'roles', 'organizations']
✓ Proceeding with a full fresh import pass to detect new CSV content


## Step 2: Run Deterministic Alignment Orchestration

Execute the complete Step 311 workflow if not recovered:

In [4]:
try:
    print("=" * 70)
    print("STARTING STEP 311 AUTOMATED DISAMBIGUATION ORCHESTRATION")
    print("=" * 70)
    
    orchestrator = Step311Orchestrator()
    projections_dict = orchestrator.run()
    run_source = "fresh_run_after_recovery" if recovered_projections is not None else "fresh_run"
    
    print("\n" + "=" * 70)
    print("ORCHESTRATION COMPLETE")
    print("=" * 70)
    
    print(f"\nRun source: {run_source}")
    print(f"Output directory: {PHASE_DIR}")
    print(f"Projections built: {list(projections_dict.keys())}")
    
except KeyboardInterrupt:
    print("\n⚠ Notebook interrupted by user")
    print("Checkpoint saved. Rerun to resume from checkpoint.")
    raise
except Exception as e:
    print(f"\n✗ Error during orchestration: {e}")
    import traceback
    traceback.print_exc()
    raise

STARTING STEP 311 AUTOMATED DISAMBIGUATION ORCHESTRATION


c:\workspace\git\borgnetzwerk\speaker-mining\speakermining\src\process\entity_disambiguation\normalization.py:119: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed = pd.to_datetime(text, dayfirst=True, errors="coerce")
c:\workspace\git\borgnetzwerk\speaker-mining\speakermining\src\process\entity_disambiguation\normalization.py:119: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed = pd.to_datetime(text, dayfirst=True, errors="coerce")
c:\workspace\git\borgnetzwerk\speaker-mining\speakermining\src\process\entity_disambiguation\normalization.py:119: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed = pd.to_datetime(text, dayfirst=True, errors="coerce")
c:\workspace\git\borgn


ORCHESTRATION COMPLETE

Run source: fresh_run_after_recovery
Output directory: C:\workspace\git\borgnetzwerk\speaker-mining\data\31_entity_disambiguation
Projections built: ['broadcasting_programs', 'series', 'episodes', 'persons', 'topics', 'roles', 'organizations']


## Step 3: Verify Output and Display Summary

Show output file sizes and row counts per core class:

In [5]:
print("\nOUTPUT VERIFICATION")
print("=" * 70)

summary_data = []

for core_class in CORE_CLASSES:
    csv_path = get_aligned_csv_path(core_class)
    
    if csv_path.exists():
        df = projections_dict.get(core_class)
        if df is not None and not df.empty:
            size_mb = csv_path.stat().st_size / (1024 * 1024)
            summary_data.append({
                "Core Class": core_class,
                "Row Count": len(df),
                "Column Count": len(df.columns),
                "File Size (MB)": f"{size_mb:.2f}",
                "Status": "✓ Written"
            })
        else:
            summary_data.append({
                "Core Class": core_class,
                "Row Count": 0,
                "Column Count": 0,
                "File Size (MB)": "0.00",
                "Status": "⊘ Empty"
            })
    else:
        summary_data.append({
            "Core Class": core_class,
            "Row Count": "N/A",
            "Column Count": "N/A",
            "File Size (MB)": "N/A",
            "Status": "✗ Missing"
        })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

print(f"\nNext Step: Step 312 Manual Reconciliation in OpenRefine")
print(f"See: documentation/31_entity_disambiguation/312_manual_reconciliation_specification.md")
print(f"\nRun completed at: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")


OUTPUT VERIFICATION


,Core Class,Row Count,Column Count,File Size (MB),Status
0,broadcasting_programs,1367,29,0.91,✓ Written
1,series,475,32,0.32,✓ Written
2,episodes,25416,36,19.04,✓ Written
3,persons,17402,40,12.91,✓ Written
4,topics,1,31,0.00,✓ Written
5,roles,0,0,0.00,⊘ Empty
6,organizations,108,32,0.17,✓ Written



Next Step: Step 312 Manual Reconciliation in OpenRefine
See: documentation/31_entity_disambiguation/312_manual_reconciliation_specification.md

Run completed at: 2026-04-12 10:02:03 UTC


## Step 4: Inspect Produced Projections

Load all projection artifacts from the projections folder and expose them as DataFrames.

In [6]:
projection_files = {
    core_class: get_aligned_csv_path(core_class)
    for core_class in CORE_CLASSES
}

projections_loaded = {}

for core_class, path in projection_files.items():
    if path.exists():
        projections_loaded[core_class] = pd.read_csv(path)
    else:
        projections_loaded[core_class] = pd.DataFrame()

broadcasting_programs_df = projections_loaded["broadcasting_programs"]
series_df = projections_loaded["series"]
episodes_df = projections_loaded["episodes"]
persons_df = projections_loaded["persons"]
topics_df = projections_loaded["topics"]
roles_df = projections_loaded["roles"]
organizations_df = projections_loaded["organizations"]

print(f"Projection root: {PROJECTION_ROOT}")
for core_class in CORE_CLASSES:
    path = projection_files[core_class]
    rows = len(projections_loaded[core_class])
    print(f"{core_class}: {rows} rows ({path})")

C:\Users\timwi\AppData\Local\Temp\ipykernel_32468\2715504959.py:10: DtypeWarning: Columns (0: source_zdf_value, 1: source_wikidata_value, 2: wikidata_claim_properties, 3: wikidata_property_counts_json, 4: wikidata_p31_qids, 5: wikidata_p179_qids, 6: wikidata_p921_qids, 7: wikidata_p361_qids) have mixed types. Specify dtype option on import or set low_memory=False.
  projections_loaded[core_class] = pd.read_csv(path)


Projection root: C:\workspace\git\borgnetzwerk\speaker-mining\data\31_entity_disambiguation\projections
broadcasting_programs: 1367 rows (C:\workspace\git\borgnetzwerk\speaker-mining\data\31_entity_disambiguation\projections\aligned_broadcasting_programs.csv)
series: 475 rows (C:\workspace\git\borgnetzwerk\speaker-mining\data\31_entity_disambiguation\projections\aligned_series.csv)
episodes: 25416 rows (C:\workspace\git\borgnetzwerk\speaker-mining\data\31_entity_disambiguation\projections\aligned_episodes.csv)
persons: 17402 rows (C:\workspace\git\borgnetzwerk\speaker-mining\data\31_entity_disambiguation\projections\aligned_persons.csv)
topics: 1 rows (C:\workspace\git\borgnetzwerk\speaker-mining\data\31_entity_disambiguation\projections\aligned_topics.csv)
roles: 0 rows (C:\workspace\git\borgnetzwerk\speaker-mining\data\31_entity_disambiguation\projections\aligned_roles.csv)
organizations: 108 rows (C:\workspace\git\borgnetzwerk\speaker-mining\data\31_entity_disambiguation\projections

### Broadcasting Programs Projection

In [7]:
broadcasting_programs_df

,alignment_unit_id,core_class,broadcasting_program_key,episode_key,source_zdf_value,source_wikidata_value,source_fernsehserien_value,deterministic_alignment_status,deterministic_alignment_score,deterministic_alignment_method,...,wikidata_claim_property_count,wikidata_claim_statement_count,wikidata_property_counts_json,wikidata_p31_qids,wikidata_p179_qids,wikidata_p106_qids,wikidata_p39_qids,wikidata_p921_qids,wikidata_p527_qids,wikidata_p361_qids
0,program::Markus Lanz,broadcasting_programs,Markus Lanz,NaN,Markus Lanz,NaN,NaN,aligned,1.0,seed_broadcasting_program,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,program::Hart aber fair,broadcasting_programs,Hart aber fair,NaN,Hart aber fair,NaN,NaN,aligned,1.0,seed_broadcasting_program,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,program::Maischberger,broadcasting_programs,Maischberger,NaN,Maischberger,NaN,NaN,aligned,1.0,seed_broadcasting_program,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,program::Maybrit Illner,broadcasting_programs,Maybrit Illner,NaN,Maybrit Illner,NaN,NaN,aligned,1.0,seed_broadcasting_program,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,program::Caren Miosga,broadcasting_programs,Caren Miosga,NaN,Caren Miosga,NaN,NaN,aligned,1.0,seed_broadcasting_program,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1362,wd_broadcasting_programs::Q1269988,broadcasting_programs,NaN,NaN,NaN,MausLive,NaN,unresolved,0.0,seed_wikidata_broadcasting_program,...,5.0,5.0,"{""P2671"": 1, ""P31"": 1, ""P449"": 1, ""P495"": 1, ""...",Q1555508,NaN,NaN,NaN,NaN,NaN,NaN
1363,wd_broadcasting_programs::Q137807876,broadcasting_programs,NaN,NaN,NaN,Satire Deluxe,NaN,unresolved,0.0,seed_wikidata_broadcasting_program,...,4.0,5.0,"{""P31"": 1, ""P449"": 2, ""P495"": 1, ""P973"": 1}",Q1555508,NaN,NaN,NaN,NaN,NaN,NaN
1364,wd_broadcasting_programs::Q1571810,broadcasting_programs,NaN,NaN,NaN,Hallo Ü-Wagen,NaN,unresolved,0.0,seed_wikidata_broadcasting_program,...,10.0,10.0,"{""P17"": 1, ""P18"": 1, ""P2671"": 1, ""P272"": 1, ""P...",Q1555508,NaN,NaN,NaN,NaN,NaN,NaN
1365,wd_broadcasting_programs::Q185015,broadcasting_programs,NaN,NaN,NaN,ZeitZeichen,NaN,unresolved,0.0,seed_wikidata_broadcasting_program,...,11.0,14.0,"{""P136"": 1, ""P1476"": 1, ""P227"": 1, ""P2671"": 1,...",Q1555508,NaN,NaN,NaN,NaN,NaN,NaN


### Series Projection

In [8]:
series_df

,alignment_unit_id,core_class,broadcasting_program_key,episode_key,source_zdf_value,source_wikidata_value,source_fernsehserien_value,deterministic_alignment_status,deterministic_alignment_score,deterministic_alignment_method,...,wikidata_claim_property_count,wikidata_claim_statement_count,wikidata_property_counts_json,wikidata_p31_qids,wikidata_p179_qids,wikidata_p106_qids,wikidata_p39_qids,wikidata_p921_qids,wikidata_p527_qids,wikidata_p361_qids
0,wd_series_from_broadcasting::Q102021344,series,NaN,NaN,NaN,Forsthaus Falkenau/Staffel 1,NaN,unresolved,0.0,seed_wikidata_series_from_broadcasting_program,...,5,5,"{""P179"": 1, ""P2671"": 1, ""P31"": 1, ""P364"": 1, ""...",Q3464665,Q878668,NaN,NaN,NaN,NaN,NaN
1,wd_series_from_broadcasting::Q102021350,series,NaN,NaN,NaN,Forsthaus Falkenau/Staffel 2,NaN,unresolved,0.0,seed_wikidata_series_from_broadcasting_program,...,5,5,"{""P179"": 1, ""P2671"": 1, ""P31"": 1, ""P364"": 1, ""...",Q3464665,Q878668,NaN,NaN,NaN,NaN,NaN
2,wd_series_from_broadcasting::Q102021358,series,NaN,NaN,NaN,Forsthaus Falkenau/Staffel 3,NaN,unresolved,0.0,seed_wikidata_series_from_broadcasting_program,...,5,5,"{""P179"": 1, ""P2671"": 1, ""P31"": 1, ""P364"": 1, ""...",Q3464665,Q878668,NaN,NaN,NaN,NaN,NaN
3,wd_series_from_broadcasting::Q102021365,series,NaN,NaN,NaN,Forsthaus Falkenau/Staffel 4,NaN,unresolved,0.0,seed_wikidata_series_from_broadcasting_program,...,5,5,"{""P179"": 1, ""P2671"": 1, ""P31"": 1, ""P364"": 1, ""...",Q3464665,Q878668,NaN,NaN,NaN,NaN,NaN
4,wd_series_from_broadcasting::Q102021372,series,NaN,NaN,NaN,Forsthaus Falkenau/Staffel 5,NaN,unresolved,0.0,seed_wikidata_series_from_broadcasting_program,...,5,5,"{""P179"": 1, ""P2671"": 1, ""P31"": 1, ""P364"": 1, ""...",Q3464665,Q878668,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
470,wd_series::Q895070,series,NaN,NaN,NaN,Boulevard Bio,NaN,unresolved,0.0,seed_wikidata_series,...,20,20,"{""P1113"": 1, ""P11460"": 1, ""P12096"": 1, ""P136"":...",Q15416,NaN,NaN,NaN,NaN,NaN,NaN
471,wd_series::Q98179809,series,NaN,NaN,NaN,"Maischberger. Die Woche, Staffel 18",NaN,unresolved,0.0,seed_wikidata_series,...,8,13,"{""P17"": 1, ""P179"": 1, ""P31"": 1, ""P364"": 1, ""P3...",Q3464665,Q1330010,NaN,NaN,NaN,Q98179825|Q98380352|Q98555926|Q98925525|Q98925...,NaN
472,wd_series::Q98536212,series,NaN,NaN,NaN,"Maybrit Illner, Staffel 22",NaN,unresolved,0.0,seed_wikidata_series,...,8,11,"{""P179"": 1, ""P31"": 1, ""P364"": 1, ""P371"": 1, ""P...",Q3464665,Q1914407,NaN,NaN,NaN,Q98536217|Q98918664|Q99194102|Q99461986,NaN
473,wd_series::Q98918765,series,NaN,NaN,NaN,"Hart aber fair, Staffel 21",NaN,unresolved,0.0,seed_wikidata_series,...,9,12,"{""P136"": 1, ""P179"": 1, ""P31"": 1, ""P364"": 1, ""P...",Q3464665,Q1587023,NaN,NaN,NaN,Q98918780|Q98923505|Q98930943|Q99232268,NaN


### Episodes Projection

In [9]:
episodes_df

,alignment_unit_id,core_class,broadcasting_program_key,episode_key,source_zdf_value,source_wikidata_value,source_fernsehserien_value,deterministic_alignment_status,deterministic_alignment_score,deterministic_alignment_method,...,wikidata_claim_property_count,wikidata_claim_statement_count,wikidata_property_counts_json,wikidata_p31_qids,wikidata_p179_qids,wikidata_p106_qids,wikidata_p39_qids,wikidata_p921_qids,wikidata_p527_qids,wikidata_p361_qids
0,zdf::ep_a371a3777018,episodes,Markus Lanz 03.06.2008,ep_a371a3777018,03.06.2008,NaN,NaN,unresolved,0.0,seed_zdf_episode,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,zdf::ep_36337a9dfb46,episodes,Markus Lanz 04.06.2008,ep_36337a9dfb46,04.06.2008,NaN,NaN,unresolved,0.0,seed_zdf_episode,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,zdf::ep_145cdbbd9501,episodes,Markus Lanz 05.06.2008,ep_145cdbbd9501,05.06.2008,NaN,NaN,unresolved,0.0,seed_zdf_episode,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,zdf::ep_75f5025ddda9,episodes,Markus Lanz 10.06.2008,ep_75f5025ddda9,10.06.2008,NaN,NaN,unresolved,0.0,seed_zdf_episode,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,zdf::ep_e102c24f59df,episodes,Markus Lanz 18.06.2008,ep_e102c24f59df,18.06.2008,NaN,NaN,unresolved,0.0,seed_zdf_episode,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25411,wd::Q98923505,episodes,Q1587023,Q98923505,NaN,Wahlkampf mit allen Mitteln: Zerbricht Amerika...,NaN,unresolved,0.0,seed_wikidata_episode,...,12.0,15.0,"{""P1476"": 1, ""P155"": 1, ""P156"": 1, ""P179"": 1, ...",Q21191270,Q1587023,NaN,NaN,Q22923830,NaN,NaN
25412,wd::Q98930943,episodes,Q1587023,Q98930943,NaN,Streit um Corona-Verbote: Wie viel Freiheit is...,NaN,unresolved,0.0,seed_wikidata_episode,...,12.0,16.0,"{""P1476"": 1, ""P155"": 1, ""P156"": 1, ""P179"": 1, ...",Q21191270,Q1587023,NaN,NaN,Q81068910,NaN,NaN
25413,wd::Q99194102,episodes,Q1914407,Q99194102,NaN,Merkels Russland-Dilemma – ratlos zwischen Put...,NaN,unresolved,0.0,seed_wikidata_episode,...,13.0,20.0,"{""P1476"": 1, ""P155"": 1, ""P156"": 1, ""P179"": 1, ...",Q21191270,Q1914407,NaN,NaN,Q83168|Q97384381|Q98640410,NaN,NaN
25414,wd::Q99232268,episodes,Q1587023,Q99232268,NaN,Vergiftete Beziehungen: Wie umgehen mit Putins...,NaN,unresolved,0.0,seed_wikidata_episode,...,12.0,18.0,"{""P1476"": 1, ""P155"": 1, ""P179"": 1, ""P31"": 1, ""...",Q21191270,Q1587023,NaN,NaN,Q21644350|Q83168|Q98640410,NaN,NaN


### Persons Projection

In [10]:
persons_df

,alignment_unit_id,core_class,broadcasting_program_key,episode_key,source_zdf_value,source_wikidata_value,source_fernsehserien_value,deterministic_alignment_status,deterministic_alignment_score,deterministic_alignment_method,...,wikidata_claim_property_count,wikidata_claim_statement_count,wikidata_property_counts_json,wikidata_p31_qids,wikidata_p179_qids,wikidata_p106_qids,wikidata_p39_qids,wikidata_p921_qids,wikidata_p527_qids,wikidata_p361_qids
0,fs::https://www.fernsehserien.de/international...,persons,Der Internationale Frühschoppen,https://www.fernsehserien.de/internationaler-f...,NaN,NaN,-Andrej Gurkov,unresolved,0.00,seed_fernsehserien_episode_guest,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,fs::https://www.fernsehserien.de/markus-lanz/f...,persons,Markus Lanz,https://www.fernsehserien.de/markus-lanz/folge...,NaN,NaN,-Andrew Denison,unresolved,0.00,seed_fernsehserien_episode_guest,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,fs::https://www.fernsehserien.de/scobel/folgen...,persons,scobel,https://www.fernsehserien.de/scobel/folgen/32-...,NaN,NaN,-psychologie,unresolved,0.00,seed_fernsehserien_episode_guest,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,fs::https://www.fernsehserien.de/scobel/folgen...,persons,scobel,https://www.fernsehserien.de/scobel/folgen/21-...,NaN,NaN,3sat,unresolved,0.00,seed_fernsehserien_episode_guest,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ep_1c0ed9e4af9c:pm_6f28c4ce016e,persons,Markus Lanz 27.01.2015,ep_1c0ed9e4af9c,Aaron ALTARAS,NaN,Aaron Altaras,aligned,0.85,name_exact_zdf_fernsehserien,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17397,fs::https://www.fernsehserien.de/markus-lanz/f...,persons,Markus Lanz,https://www.fernsehserien.de/markus-lanz/folge...,NaN,NaN,„Ingrid,unresolved,0.00,seed_fernsehserien_episode_guest,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17398,fs::https://www.fernsehserien.de/markus-lanz/f...,persons,Markus Lanz,https://www.fernsehserien.de/markus-lanz/folge...,NaN,NaN,„Onkel Fisch“,unresolved,0.00,seed_fernsehserien_episode_guest,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17399,fs::https://www.fernsehserien.de/maischberger-...,persons,Maischberger,https://www.fernsehserien.de/maischberger-ard/...,NaN,NaN,„Süddeutsche Zeitung“,unresolved,0.00,seed_fernsehserien_episode_guest,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17400,fs::https://www.fernsehserien.de/markus-lanz/f...,persons,Markus Lanz,https://www.fernsehserien.de/markus-lanz/folge...,NaN,NaN,„The Boss Hoss“,unresolved,0.00,seed_fernsehserien_episode_guest,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Topics Projection

In [11]:
topics_df

,alignment_unit_id,core_class,broadcasting_program_key,episode_key,source_zdf_value,source_wikidata_value,source_fernsehserien_value,deterministic_alignment_status,deterministic_alignment_score,deterministic_alignment_method,...,wikidata_claim_property_count,wikidata_claim_statement_count,wikidata_property_counts_json,wikidata_p31_qids,wikidata_p179_qids,wikidata_p106_qids,wikidata_p39_qids,wikidata_p921_qids,wikidata_p527_qids,wikidata_p361_qids
0,wd_topics::Q15860072,topics,NaN,NaN,NaN,Russisch-Ukrainischer Krieg,NaN,unresolved,0.0,seed_wikidata_topic,...,54,98,"{""P10804"": 1, ""P1151"": 1, ""P12126"": 1, ""P1269""...",Q11422542|Q11514315|Q1168287|Q198|Q474138|Q788...,NaN,NaN,NaN,NaN,Q110254389|Q110999040|Q113495737|Q114096220|Q1...,Q17510383|Q1975306


### Roles Projection

In [12]:
roles_df

,alignment_unit_id,core_class,broadcasting_program_key,episode_key,source_zdf_value,source_wikidata_value,source_fernsehserien_value,deterministic_alignment_status,deterministic_alignment_score,deterministic_alignment_method,...,wikidata_claim_property_count,wikidata_claim_statement_count,wikidata_property_counts_json,wikidata_p31_qids,wikidata_p179_qids,wikidata_p106_qids,wikidata_p39_qids,wikidata_p921_qids,wikidata_p527_qids,wikidata_p361_qids


### Organizations Projection

In [13]:
organizations_df

,alignment_unit_id,core_class,broadcasting_program_key,episode_key,source_zdf_value,source_wikidata_value,source_fernsehserien_value,deterministic_alignment_status,deterministic_alignment_score,deterministic_alignment_method,...,wikidata_claim_property_count,wikidata_claim_statement_count,wikidata_property_counts_json,wikidata_p31_qids,wikidata_p179_qids,wikidata_p106_qids,wikidata_p39_qids,wikidata_p921_qids,wikidata_p527_qids,wikidata_p361_qids
0,wd_organizations::Q1000950,organizations,NaN,NaN,NaN,WDR 5,NaN,unresolved,0.0,seed_wikidata_organization,...,15,15,"{""P127"": 1, ""P137"": 1, ""P154"": 1, ""P17"": 1, ""P...",Q14350,NaN,NaN,NaN,NaN,NaN,NaN
1,wd_organizations::Q1022,organizations,NaN,NaN,NaN,Stuttgart,NaN,unresolved,0.0,seed_wikidata_organization,...,206,431,"{""P1017"": 1, ""P10280"": 1, ""P10291"": 1, ""P10321...",Q106517174|Q1066984|Q14784328|Q1549591|Q232751...,NaN,NaN,NaN,NaN,NaN,Q451619|Q8172
2,wd_organizations::Q102734,organizations,NaN,NaN,NaN,Rote Armee Fraktion,NaN,unresolved,0.0,seed_wikidata_organization,...,61,78,"{""P10234"": 1, ""P112"": 4, ""P1142"": 3, ""P1151"": ...",Q17127659|Q17149090|Q20642019|Q7933191,NaN,NaN,NaN,NaN,NaN,NaN
3,wd_organizations::Q1034,organizations,NaN,NaN,NaN,Biel/Bienne,NaN,unresolved,0.0,seed_wikidata_organization,...,83,166,"{""P1036"": 1, ""P10397"": 1, ""P1082"": 44, ""P11514...",Q54935504|Q70208,NaN,NaN,NaN,NaN,NaN,NaN
4,wd_organizations::Q106648083,organizations,NaN,NaN,NaN,Isabella von Ägypten,NaN,unresolved,0.0,seed_wikidata_organization,...,11,12,"{""P1191"": 1, ""P144"": 2, ""P2047"": 1, ""P227"": 1,...",Q2635894,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,wd_organizations::Q702486,organizations,NaN,NaN,NaN,Universität Trier,NaN,unresolved,0.0,seed_wikidata_organization,...,60,110,"{""P10283"": 1, ""P11686"": 1, ""P11961"": 1, ""P1197...",Q5028741|Q875538,NaN,NaN,NaN,NaN,Q110857159|Q110859627|Q124695027|Q134355274|Q1...,NaN
104,wd_organizations::Q821274,organizations,NaN,NaN,NaN,Berlin Debating Union,NaN,unresolved,0.0,seed_wikidata_organization,...,8,8,"{""P112"": 1, ""P1454"": 1, ""P159"": 1, ""P17"": 1, ""...",Q1181396,NaN,NaN,NaN,NaN,NaN,NaN
105,wd_organizations::Q82229,organizations,NaN,NaN,NaN,Rocket Internet SE,NaN,unresolved,0.0,seed_wikidata_organization,...,30,47,"{""P112"": 1, ""P11245"": 1, ""P127"": 1, ""P1278"": 1...",Q19252952|Q891723,NaN,NaN,NaN,NaN,NaN,Q448445|Q595622
106,wd_organizations::Q10184,organizations,NaN,NaN,NaN,Frankfurter Allgemeine Zeitung,NaN,unresolved,0.0,seed_wikidata_organization,...,70,93,"{""P101"": 1, ""P1015"": 1, ""P1042"": 1, ""P10565"": ...",Q1110794|Q43229,NaN,NaN,NaN,NaN,NaN,NaN
